In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import os
import seaborn as sns
import pickle
import math
from tqdm import tqdm
from pandas import HDFStore
from scipy.sparse.linalg import svds, eigs
import random


In [2]:
# Load training graph
if os.path.isfile('train_pos_after_eda.csv'):
    train_graph = nx.read_edgelist('train_pos_after_eda.csv', delimiter=',', create_using=nx.DiGraph(), nodetype=int)
    print(f"Graph Info:")
    print(f"Number of nodes: {train_graph.number_of_nodes()}")
    print(f"Number of edges: {train_graph.number_of_edges()}")
    print(f"Graph type: {type(train_graph)}")
else:
    print("Please run the FB_EDA.ipynb or download the files from drive")

Graph Info:
Number of nodes: 1780722
Number of edges: 7550015
Graph type: <class 'networkx.classes.digraph.DiGraph'>


In [ ]:
# ===========================
# SIMILARITY MEASURES
# ===========================

# Jaccard Distance for followees
def jaccard_for_followees(a, b):
    try:
        if len(set(train_graph.successors(a))) == 0 | len(set(train_graph.successors(b))) == 0:
            return 0
        sim = (len(set(train_graph.successors(a)).intersection(set(train_graph.successors(b))))) / \
              (len(set(train_graph.successors(a)).union(set(train_graph.successors(b)))))
    except:
        return 0
    return sim

In [ ]:
# Jaccard Distance for followers
def jaccard_for_followers(a, b):
    try:
        if len(set(train_graph.predecessors(a))) == 0 | len(set(train_graph.predecessors(b))) == 0:
            return 0
        sim = (len(set(train_graph.predecessors(a)).intersection(set(train_graph.predecessors(b))))) / \
              (len(set(train_graph.predecessors(a)).union(set(train_graph.predecessors(b)))))
        return sim
    except:
        return 0

In [ ]:
# Cosine Distance for followees
def cosine_for_followees(a, b):
    try:
        if len(set(train_graph.successors(a))) == 0 | len(set(train_graph.successors(b))) == 0:
            return 0
        sim = (len(set(train_graph.successors(a)).intersection(set(train_graph.successors(b))))) / \
              (math.sqrt(len(set(train_graph.successors(a))) * len((set(train_graph.successors(b))))))
        return sim
    except:
        return 0

In [ ]:
# Cosine Distance for followers
def cosine_for_followers(a, b):
    try:
        if len(set(train_graph.predecessors(a))) == 0 | len(set(train_graph.predecessors(b))) == 0:
            return 0
        sim = (len(set(train_graph.predecessors(a)).intersection(set(train_graph.predecessors(b))))) / \
              (math.sqrt(len(set(train_graph.predecessors(a)))) * (len(set(train_graph.predecessors(b)))))
        return sim
    except:
        return 0

In [ ]:
# ===========================
# PAGE RANK
# ===========================

# PageRank calculation and caching
if not os.path.isfile('page_rank_final.p'):
    pr = nx.pagerank(train_graph, alpha=0.85)
    pickle.dump(pr, open('page_rank_final.p', 'wb'))
else:
    pr = pickle.load(open('page_rank_final.p', 'rb'))

print('PageRank min:', pr[min(pr, key=pr.get)])
print('PageRank max:', pr[max(pr, key=pr.get)])
print('PageRank mean:', float(sum(pr.values())) / len(pr))

# For imputing to nodes which are not there in Train data
mean_pr = float(sum(pr.values())) / len(pr)
print(f"Mean PageRank: {mean_pr}")

In [ ]:

# ===========================
# OTHER GRAPH FEATURES
# ===========================

# Shortest path calculation
def compute_shortest_path_length(a, b):
    p = -1
    try:
        if train_graph.has_edge(a, b):
            train_graph.remove_edge(a, b)
            p = nx.shortest_path_length(train_graph, source=a, target=b)
            train_graph.add_edge(a, b)
        else:
            p = nx.shortest_path_length(train_graph, source=a, target=b)
        return p
    except:
        return -1

In [ ]:
# Weakly connected components
wcc = list(nx.weakly_connected_components(train_graph))

def belongs_to_same_wcc(a, b):
    index = []
    if train_graph.has_edge(b, a):
        return 1
    if train_graph.has_edge(a, b):
        for i in wcc:
            if a in i:
                index = i
                break
        if (b in index):
            train_graph.remove_edge(a, b)
            if compute_shortest_path_length(a, b) == -1:
                train_graph.add_edge(a, b)
                return 0
            else:
                train_graph.add_edge(a, b)
                return 1
        else:
            return 0
    else:
        for i in wcc:
            if a in i:
                index = i
                break
        if (b in index):
            return 1
        else:
            return 0


In [ ]:
# Adamic/Adar Index
def calc_adar_in(a, b):
    sum_val = 0
    try:
        n = list(set(train_graph.successors(a)).intersection(set(train_graph.successors(b))))
        if len(n) != 0:
            for i in n:
                sum_val = sum_val + (1 / np.log10(len(list(train_graph.predecessors(i)))))
            return sum_val
        else:
            return 0
    except:
        return 0

In [ ]:
# Follows back
def follows_back(a, b):
    if train_graph.has_edge(b, a):
        return 1
    else:
        return 0


In [ ]:
# ===========================
# KATZ CENTRALITY
# ===========================

if not os.path.isfile('katz_final.p'):
    katz = nx.katz_centrality(train_graph, alpha=0.005, beta=1)
    pickle.dump(katz, open('katz_final.p', 'wb'))
else:
    katz = pickle.load(open('katz_final.p', 'rb'))

print('Katz min:', katz[min(katz, key=katz.get)])
print('Katz max:', katz[max(katz, key=katz.get)])
print('Katz mean:', float(sum(katz.values())) / len(katz))

mean_katz = float(sum(katz.values())) / len(katz)
print(f"Mean Katz: {mean_katz}")

In [ ]:
# ===========================
# HITS ALGORITHM
# ===========================

if not os.path.isfile('hits_final.p'):
    hits = nx.hits(train_graph, max_iter=100, tol=1e-08, nstart=None, normalized=True)
    pickle.dump(hits, open('hits_final.p', 'wb'))
else:
    hits = pickle.load(open('hits_final.p', 'rb'))

print('HITS Hubs min:', hits[0][min(hits[0], key=hits[0].get)])
print('HITS Hubs max:', hits[0][max(hits[0], key=hits[0].get)])
print('HITS Hubs mean:', float(sum(hits[0].values())) / len(hits[0]))

print('\nHITS Authority Scores:')
print('HITS Authority min:', hits[1][min(hits[1], key=hits[1].get)])
print('HITS Authority max:', hits[1][max(hits[1], key=hits[1].get)])
print('HITS Authority mean:', float(sum(hits[1].values())) / len(hits[1]))

In [ ]:
# ===========================
# PREFERENTIAL ATTACHMENT
# ===========================

def preferential_attachment(a, b):
    """
    Preferential Attachment: Product of degrees of source and destination nodes
    """
    try:
        score = len(list(train_graph.successors(a))) * len(list(train_graph.successors(b)))
        return score
    except:
        return 0

In [ ]:
# ===========================
# DATA SAMPLING
# ===========================

# Sample data for processing
if os.path.isfile('train_after_eda.csv'):
    n_train = 15100028
    s = 100000
    skip_train = sorted(random.sample(range(1, n_train + 1), n_train - s))

if os.path.isfile('test_after_eda.csv'):
    n_test = 3775006
    s = 50000
    skip_test = sorted(random.sample(range(1, n_test + 1), n_test - s))

print("Number of rows in the train data file:", n_train)
print("Number of rows we are going to eliminate in train data are", len(skip_train))
print("Number of rows in the test data file:", n_test)
print("Number of rows we are going to eliminate in test data are", len(skip_test))


In [ ]:
# ===========================
# FEATURE ENGINEERING FUNCTIONS
# ===========================

def compute_features_stage1(df_final):
    """Calculate basic graph features"""
    num_followers_s = []
    num_followees_s = []
    num_followers_d = []
    num_followees_d = []
    inter_followers = []
    inter_followees = []
    
    for i, row in tqdm(df_final.iterrows(), total=len(df_final)):
        try:
            s1 = set(train_graph.predecessors(row['source_node']))
            s2 = set(train_graph.successors(row['source_node']))
        except:
            s1 = set()
            s2 = set()
        try:
            d1 = set(train_graph.predecessors(row['destination_node']))
            d2 = set(train_graph.successors(row['destination_node']))
        except:
            d1 = set()
            d2 = set()
            
        num_followers_s.append(len(s1))
        num_followees_s.append(len(s2))
        num_followers_d.append(len(d1))
        num_followees_d.append(len(d2))
        inter_followers.append(len(s1.intersection(d1)))
        inter_followees.append(len(s2.intersection(d2)))
    
    return num_followers_s, num_followees_s, num_followers_d, num_followees_d, inter_followers, inter_followees


In [ ]:

# SVD FEATURES
# ===========================

def svd_features(x, S):
    """Get SVD features for a node"""
    try:
        z = sadj_dict[x]
        return S[z]
    except:
        return [0, 0, 0, 0, 0, 0]

# Create adjacency matrix for SVD
sadj_col = sorted(train_graph.nodes())
sadj_dict = {val: idx for idx, val in enumerate(sadj_col)}
Adj = nx.adjacency_matrix(train_graph, nodelist=sorted(train_graph.nodes())).asfptype()

# Compute SVD
U, s, V = svds(Adj, k=6)
print('Adjacency matrix Shape:', Adj.shape)
print('U Shape:', U.shape)
print('V Shape:', V.shape)
print('s Shape:', s.shape)

In [ ]:
# ===========================
# SVD DOT PRODUCT FEATURE
# ===========================

def compute_svd_dot(df):
    """Compute dot product between source and destination SVD features"""
    svd_dot_u = []
    svd_dot_v = []
    
    for i, row in tqdm(df.iterrows(), total=len(df)):
        source_svd_u = svd_features(row['source_node'], U)
        dest_svd_u = svd_features(row['destination_node'], U)
        dot_u = np.dot(source_svd_u, dest_svd_u)
        svd_dot_u.append(dot_u)
        
        source_svd_v = svd_features(row['source_node'], V.T)
        dest_svd_v = svd_features(row['destination_node'], V.T)
        dot_v = np.dot(source_svd_v, dest_svd_v)
        svd_dot_v.append(dot_v)
    
    return svd_dot_u, svd_dot_v

In [ ]:
# ===========================
# WEIGHT FEATURES
# ===========================

# Calculate weight features
Weight_in = {}
Weight_out = {}

for i in tqdm(train_graph.nodes()):
    s1 = set(train_graph.predecessors(i))
    w_in = 1.0 / (np.sqrt(1 + len(s1)))
    Weight_in[i] = w_in
    
    s2 = set(train_graph.successors(i))
    w_out = 1.0 / (np.sqrt(1 + len(s2)))
    Weight_out[i] = w_out

# For imputing with mean
mean_weight_in = np.mean(list(Weight_in.values()))
mean_weight_out = np.mean(list(Weight_out.values()))


In [ ]:
# ===========================
# MAIN FEATURE ENGINEERING PIPELINE
# ===========================

# Load sample data
df_final_train = pd.read_csv('train_after_eda.csv', skiprows=skip_train, names=['source_node', 'destination_node'])
df_final_train['indicator_link'] = pd.read_csv('train_y.csv', skiprows=skip_train, names=['indicator_link'])['indicator_link']

df_final_test = pd.read_csv('test_after_eda.csv', skiprows=skip_test, names=['source_node', 'destination_node'])
df_final_test['indicator_link'] = pd.read_csv('test_y.csv', skiprows=skip_test, names=['indicator_link'])['indicator_link']

print("Train matrix size:", df_final_train.shape)
print("Test matrix size:", df_final_test.shape)

In [ ]:
# ===========================
# STAGE 1: Basic Graph Features
# ===========================

if not os.path.isfile('final_storage_stage1.h5'):
    print("Computing Stage 1 Features...")
    (df_final_train['num_followers_s'], df_final_train['num_followees_s'], 
     df_final_train['num_followers_d'], df_final_train['num_followees_d'], 
     df_final_train['inter_followers'], df_final_train['inter_followees']) = compute_features_stage1(df_final_train)
    
    (df_final_test['num_followers_s'], df_final_test['num_followees_s'], 
     df_final_test['num_followers_d'], df_final_test['num_followees_d'], 
     df_final_test['inter_followers'], df_final_test['inter_followees']) = compute_features_stage1(df_final_test)
    
    hdf = HDFStore('final_storage_stage1.h5')
    hdf.put('train_df', df_final_train, format='table', data_columns=True)
    hdf.put('test_df', df_final_test, format='table', data_columns=True)
    hdf.close()
else:
    df_final_train = pd.read_hdf('final_storage_stage1.h5', 'train_df', mode='r')
    df_final_test = pd.read_hdf('final_storage_stage1.h5', 'test_df', mode='r')


In [ ]:
# ===========================
# STAGE 2: Advanced Graph Features
# ===========================

if not os.path.isfile('final_storage_stage2.h5'):
    print("Computing Stage 2 Features...")
    
    # Adamic/Adar Index
    df_final_train['adar_index'] = df_final_train.apply(lambda row: calc_adar_in(row['source_node'], row['destination_node']), axis=1)
    df_final_test['adar_index'] = df_final_test.apply(lambda row: calc_adar_in(row['source_node'], row['destination_node']), axis=1)
    
    # Follows back
    df_final_train['follows_back'] = df_final_train.apply(lambda row: follows_back(row['source_node'], row['destination_node']), axis=1)
    df_final_test['follows_back'] = df_final_test.apply(lambda row: follows_back(row['source_node'], row['destination_node']), axis=1)
    
    # Same weakly connected component
    df_final_train['same_comp'] = df_final_train.apply(lambda row: belongs_to_same_wcc(row['source_node'], row['destination_node']), axis=1)
    df_final_test['same_comp'] = df_final_test.apply(lambda row: belongs_to_same_wcc(row['source_node'], row['destination_node']), axis=1)
    
    # Shortest path
    df_final_train['shortest_path'] = df_final_train.apply(lambda row: compute_shortest_path_length(row['source_node'], row['destination_node']), axis=1)
    df_final_test['shortest_path'] = df_final_test.apply(lambda row: compute_shortest_path_length(row['source_node'], row['destination_node']), axis=1)
    
    # Preferential Attachment
    df_final_train['preferential_attachment'] = df_final_train.apply(lambda row: preferential_attachment(row['source_node'], row['destination_node']), axis=1)
    df_final_test['preferential_attachment'] = df_final_test.apply(lambda row: preferential_attachment(row['source_node'], row['destination_node']), axis=1)
    
    hdf = HDFStore('final_storage_stage2.h5')
    hdf.put('train_df', df_final_train, format='table', data_columns=True)
    hdf.put('test_df', df_final_test, format='table', data_columns=True)
    hdf.close()
else:
    df_final_train = pd.read_hdf('final_storage_stage2.h5', 'train_df', mode='r')
    df_final_test = pd.read_hdf('final_storage_stage2.h5', 'test_df', mode='r')


In [ ]:
# ===========================
# STAGE 3: Centrality and Weight Features
# ===========================

if not os.path.isfile('final_storage_stage3.h5'):
    print("Computing Stage 3 Features...")
    
    # Weight features
    df_final_train['weight_in'] = df_final_train.destination_node.apply(lambda x: Weight_in.get(x, mean_weight_in))
    df_final_train['weight_out'] = df_final_train.source_node.apply(lambda x: Weight_out.get(x, mean_weight_out))
    df_final_test['weight_in'] = df_final_test.destination_node.apply(lambda x: Weight_in.get(x, mean_weight_in))
    df_final_test['weight_out'] = df_final_test.source_node.apply(lambda x: Weight_out.get(x, mean_weight_out))
    
    # Weight combinations
    df_final_train['weight_f1'] = df_final_train.weight_in + df_final_train.weight_out
    df_final_train['weight_f2'] = df_final_train.weight_in * df_final_train.weight_out
    df_final_train['weight_f3'] = (2 * df_final_train.weight_in + 1 * df_final_train.weight_out)
    df_final_train['weight_f4'] = (1 * df_final_train.weight_in + 2 * df_final_train.weight_out)
    
    df_final_test['weight_f1'] = df_final_test.weight_in + df_final_test.weight_out
    df_final_test['weight_f2'] = df_final_test.weight_in * df_final_test.weight_out
    df_final_test['weight_f3'] = (2 * df_final_test.weight_in + 1 * df_final_test.weight_out)
    df_final_test['weight_f4'] = (1 * df_final_test.weight_in + 2 * df_final_test.weight_out)
    
    # PageRank features
    df_final_train['page_rank_s'] = df_final_train.source_node.apply(lambda x: pr.get(x, mean_pr))
    df_final_train['page_rank_d'] = df_final_train.destination_node.apply(lambda x: pr.get(x, mean_pr))
    df_final_test['page_rank_s'] = df_final_test.source_node.apply(lambda x: pr.get(x, mean_pr))
    df_final_test['page_rank_d'] = df_final_test.destination_node.apply(lambda x: pr.get(x, mean_pr))
    
    # Katz centrality features
    df_final_train['katz_s'] = df_final_train.source_node.apply(lambda x: katz.get(x, mean_katz))
    df_final_train['katz_d'] = df_final_train.destination_node.apply(lambda x: katz.get(x, mean_katz))
    df_final_test['katz_s'] = df_final_test.source_node.apply(lambda x: katz.get(x, mean_katz))
    df_final_test['katz_d'] = df_final_test.destination_node.apply(lambda x: katz.get(x, mean_katz))
    
    # HITS features
    df_final_train['hubs_s'] = df_final_train.source_node.apply(lambda x: hits[0].get(x, 0))
    df_final_train['hubs_d'] = df_final_train.destination_node.apply(lambda x: hits[0].get(x, 0))
    df_final_test['hubs_s'] = df_final_test.source_node.apply(lambda x: hits[0].get(x, 0))
    df_final_test['hubs_d'] = df_final_test.destination_node.apply(lambda x: hits[0].get(x, 0))
    
    df_final_train['authorities_s'] = df_final_train.source_node.apply(lambda x: hits[1].get(x, 0))
    df_final_train['authorities_d'] = df_final_train.destination_node.apply(lambda x: hits[1].get(x, 0))
    df_final_test['authorities_s'] = df_final_test.source_node.apply(lambda x: hits[1].get(x, 0))
    df_final_test['authorities_d'] = df_final_test.destination_node.apply(lambda x: hits[1].get(x, 0))
    
    hdf = HDFStore('final_storage_stage3.h5')
    hdf.put('train_df', df_final_train, format='table', data_columns=True)
    hdf.put('test_df', df_final_test, format='table', data_columns=True)
    hdf.close()
else:
    df_final_train = pd.read_hdf('final_storage_stage3.h5', 'train_df', mode='r')
    df_final_test = pd.read_hdf('final_storage_stage3.h5', 'test_df', mode='r')


In [ ]:
# ===========================
# STAGE 4: SVD Features
# ===========================

if not os.path.isfile('final_storage_stage4.h5'):
    print("Computing Stage 4 Features...")
    
    # SVD U features
    df_final_train[['svd_u_s_1', 'svd_u_s_2', 'svd_u_s_3', 'svd_u_s_4', 'svd_u_s_5', 'svd_u_s_6']] = \
        df_final_train.source_node.apply(lambda x: svd_features(x, U)).apply(pd.Series)
    
    df_final_train[['svd_u_d_1', 'svd_u_d_2', 'svd_u_d_3', 'svd_u_d_4', 'svd_u_d_5', 'svd_u_d_6']] = \
        df_final_train.destination_node.apply(lambda x: svd_features(x, U)).apply(pd.Series)
    
    # SVD V features
    df_final_train[['svd_v_s_1', 'svd_v_s_2', 'svd_v_s_3', 'svd_v_s_4', 'svd_v_s_5', 'svd_v_s_6']] = \
        df_final_train.source_node.apply(lambda x: svd_features(x, V.T)).apply(pd.Series)
    
    df_final_train[['svd_v_d_1', 'svd_v_d_2', 'svd_v_d_3', 'svd_v_d_4', 'svd_v_d_5', 'svd_v_d_6']] = \
        df_final_train.destination_node.apply(lambda x: svd_features(x, V.T)).apply(pd.Series)
    
    # Same for test data
    df_final_test[['svd_u_s_1', 'svd_u_s_2', 'svd_u_s_3', 'svd_u_s_4', 'svd_u_s_5', 'svd_u_s_6']] = \
        df_final_test.source_node.apply(lambda x: svd_features(x, U)).apply(pd.Series)
    
    df_final_test[['svd_u_d_1', 'svd_u_d_2', 'svd_u_d_3', 'svd_u_d_4', 'svd_u_d_5', 'svd_u_d_6']] = \
        df_final_test.destination_node.apply(lambda x: svd_features(x, U)).apply(pd.Series)
    
    df_final_test[['svd_v_s_1', 'svd_v_s_2', 'svd_v_s_3', 'svd_v_s_4', 'svd_v_s_5', 'svd_v_s_6']] = \
        df_final_test.source_node.apply(lambda x: svd_features(x, V.T)).apply(pd.Series)
    
    df_final_test[['svd_v_d_1', 'svd_v_d_2', 'svd_v_d_3', 'svd_v_d_4', 'svd_v_d_5', 'svd_v_d_6']] = \
        df_final_test.destination_node.apply(lambda x: svd_features(x, V.T)).apply(pd.Series)
    
    # SVD Dot Product Features
    print("Computing SVD Dot Product Features...")
    df_final_train['svd_dot_u'], df_final_train['svd_dot_v'] = compute_svd_dot(df_final_train)
    df_final_test['svd_dot_u'], df_final_test['svd_dot_v'] = compute_svd_dot(df_final_test)
    
    hdf = HDFStore('final_storage_stage4.h5')
    hdf.put('train_df', df_final_train, format='table', data_columns=True)
    hdf.put('test_df', df_final_test, format='table', data_columns=True)
    hdf.close()
else:
    df_final_train = pd.read_hdf('final_storage_stage4.h5', 'train_df', mode='r')
    df_final_test = pd.read_hdf('final_storage_stage4.h5', 'test_df', mode='r')


In [ ]:

print("\n" + "="*60)
print("FINAL FEATURE ENGINEERING SUMMARY")
print("="*60)

print(f"Train Dataset Shape: {df_final_train.shape}")
print(f"Test Dataset Shape: {df_final_test.shape}")

print(f"\nFinal Feature List ({len(df_final_train.columns)} features):")
for i, feature in enumerate(df_final_train.columns, 1):
    print(f"{i:2d}. {feature}")

print(f"\nFeature Categories:")
print(f"- Basic Graph Features: num_followers, num_followees, inter_followers, inter_followees")
print(f"- Similarity Features: jaccard, cosine")
print(f"- Centrality Features: page_rank, katz, hubs, authorities")
print(f"- Path Features: shortest_path, same_comp")
print(f"- Advanced Features: adar_index, follows_back, preferential_attachment")
print(f"- Weight Features: weight_in, weight_out, weight_f1-f4")
print(f"- SVD Features: svd_u, svd_v (12 dimensions), svd_dot")

print(f"\n✅ All features have been successfully engineered and stored!")
print(f"📁 Final dataset saved in: final_storage_stage4.h5")

# Save to CSV for easy access
df_final_train.to_csv('final_featured_train_data.csv', index=False)
df_final_test.to_csv('final_featured_test_data.csv', index=False)

print(f"📄 CSV files also saved: final_featured_train_data.csv, final_featured_test_data.csv")